# Plan thématique d'une réunion / d'un enseignement, puis découpage de la vidéo

À partir d'une vidéo (fichier, YouTube, enregistrement Teams/Zoom), ce notebook :

1. **transcrit** la parole avec horodatage (faster-whisper) ;
2. **repère les grandes sections thématiques** (de l'ordre de 5 à 10 minutes, plus si la même notion se prolonge) ;
3. **rédige pour chaque section un thème sous forme de phrase compréhensible et une description** (LLM) ;
4. **découpe la vidéo** selon ces sections et produit des **chapitres** (fichier YouTube, vidéo avec chapitres intégrés).

Résultat visé (`plan_thematique.txt`) :

```
I. Thème : Pourquoi rester attaché à l'appel de Dieu change notre manière de vivre au quotidien (partie 1/2)
Section : [01:05 - 07:10]
Description : Résumé de la section en quelques phrases.

II. Thème : Un autre sujet, énoncé comme une phrase complète
Section : [07:10 - 15:05]
Description : Résumé de la section.
```

La partie « qui parle » est volontairement laissée de côté.

## Comment ça marche

| Étape | Méthode |
|---|---|
| Transcription | faster-whisper (GPU conseillé), audio 16 kHz passé directement en tableau numpy (évite les erreurs PyAV vues sur Colab) |
| Frontières | Le texte est regroupé en **blocs de 30 s**, transformés en vecteurs. Pour chaque point de coupure possible, on compare **3 minutes avant** et **3 minutes après** : les creux de similarité les plus marqués deviennent des frontières, avec un minimum de 5 min entre deux frontières. |
| Longueur | Durée visée par section (`TARGET_SECTION_SEC`), minimum (`MIN_SECTION_SEC`) et maximum (`MAX_SECTION_SEC`, coupure forcée au-delà). |
| Thème et description | Un **LLM** lit chaque section (en connaissant les thèmes précédents) et rédige thème + description. Il indique aussi si la section **prolonge** la précédente : elles sont alors **fusionnées** (jusqu'au maximum) ou numérotées « partie 1/2, 2/2 ». |
| Sans LLM | Repli : phrases les plus représentatives de la section (résumé extractif). Lisible, mais moins fin. |

## Où tourne quoi

- **Colab (GPU T4)** : Whisper et un LLM open source (Qwen2.5-7B en 4 bits) tournent sur le GPU. Active le GPU : *Exécution → Modifier le type d'exécution → T4*.
- **PC local (sans GPU CUDA)** : Whisper `small`/`medium` sur CPU ; pour le LLM, utilise **Ollama** (`ollama pull qwen2.5:7b-instruct`), détecté automatiquement.
- **API** (option, à activer explicitement dans la configuration) : serveur compatible OpenAI, ou API Anthropic. Le texte de la réunion quitte alors ta machine.

Chaque étape écrit son résultat dans le dossier du projet (`transcript.json`, `sections.json`, `plan_thematique.txt`, `sections_video/`) et **se saute si le résultat existe déjà**.

> **Cadre légal** : enregistrer et republier une réunion demande le consentement des participants. Pour du contenu YouTube, respecte les conditions de la plateforme et le droit d'auteur.

## Où est le code ?

Le notebook ne contient que la configuration et un appel par étape. Le code est dans le dossier [`pipeline/`](pipeline/) :

| Étape | Appel | Module |
|---|---|---|
| Configuration | `Config(...)` | `pipeline/config.py` |
| Environnement | `setup_environment` | `pipeline/environment.py` |
| Source (fichier, dossier, URL, playlist) | `resolve_sources` | `pipeline/source.py`, `pipeline/download.py` |
| Transcription | `transcribe` | `pipeline/transcription.py` |
| Sections et thèmes | `build_sections` | `pipeline/boundaries.py`, `pipeline/llm.py`, `pipeline/themes.py` |
| Texte complet | `export_full_text` | `pipeline/exports.py` |
| Découpage vidéo | `cut_video` | `pipeline/video.py` |
| Résultats | `show_results` | `pipeline/exports.py` |

Une **playlist** (ou un dossier de vidéos) est traitée vidéo par vidéo : chaque vidéo a son propre dossier de résultats dans `outputs/<playlist>/`.

## 1. Configuration
C'est la **seule cellule à modifier**.

In [ ]:
from pipeline.config import Config

cfg = Config(

    # --- Entrée ---------------------------------------------------------------
    INPUT="data/1.Création d'une nouvelle société dans Bob 50.mp4",  # fichier, dossier de vidéos (ex. une playlist déjà téléchargée), URL YouTube (vidéo ou playlist), ou "" (Colab : téléversement)
    PLAYLIST=False,         # URL watch?v=..&list=.. : True = télécharger toute la playlist (une URL /playlist?list=.. en est toujours une)
    PLAYLIST_ITEMS="",      # éléments de la playlist à traiter, ex. "1-3" ou "1,3,7" ; vide = tous
    TRANSCRIPT_FILE="",  # optionnel : .vtt / .srt Teams, Zoom... -> remplace Whisper

    # --- Étapes ---------------------------------------------------------------
    RUN_TRANSCRIBE=True,  # 1. transcription (ou import du .vtt)
    RUN_THEMES=True,  # 2. sections thématiques + thèmes + descriptions
    RUN_VIDEO=True,  # 3. découpage de la vidéo + chapitres
    FORCE=set(),  # refaire malgré le cache : {"audio", "transcribe", "themes", "video"}
    INSTALL_DEPS=True,  # installe les paquets manquants

    # --- Dossiers -------------------------------------------------------------
    WORK_DIR="outputs",  # en local : transcript.json, sections.json, plan_thematique.txt, sections_video/...
    DOWNLOAD_DIR="data",            # vidéos téléchargées depuis YouTube (Colab : <dossier de travail>/data)
    MOUNT_DRIVE=True,  # Colab : écrire dans Google Drive (résultats conservés)
    DRIVE_SUBDIR="meeting_pipeline",

    # --- Transcription --------------------------------------------------------
    LANGUAGE="fr",  # "fr", "en", ou None (détection automatique)
    MULTILINGUAL=True,  # True = langue détectée par segment (contenu mêlant FR et EN)
    WHISPER_MODEL="auto",  # auto : large-v3-turbo si GPU, medium sinon (plus fiable que small, ~2-3x plus lent ; "small" pour aller vite)
    WORD_TIMESTAMPS=True,  # horodatage mot par mot (sous-titres animés, clips calés sur les mots)
    INITIAL_PROMPT="Jésus-Christ, Saint-Esprit, CMCI, CMFI, ZTF, convention, jeûne, réveil, repentance, fr Theodore Andoseh, fr Zach, fr Zacharias Tanee Fomum.",  # noms propres, lieux, jargon : à compléter

    # --- Découpage thématique -------------------------------------------------
    MIN_SECTION_SEC=300,  # durée minimale d'une section (5 min)
    TARGET_SECTION_SEC=480,  # durée visée (8 min) : augmente-la pour des sections plus longues
    MAX_SECTION_SEC=1200,  # au-delà (20 min), coupure forcée ; limite aussi les fusions
    BLOCK_SEC=30,  # taille des blocs de texte comparés
    TOPIC_WINDOW_SEC=180,  # durée comparée avant/après un point de coupure
    TOPIC_STRICTNESS=0.0,  # > 0 : ne garde que les ruptures les plus nettes (sections plus longues)
    EMBED_MODEL="BAAI/bge-m3",  # multilingue ; plus léger : "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    # --- Rédaction des thèmes et descriptions (LLM) --------------------------
    LLM_BACKEND="auto",  # "auto" | "transformers" (GPU) | "openai" (Ollama, LM Studio, API compatible) | "anthropic" | "none"
    OUTPUT_LANGUAGE="français",
    GLOSSARY="Jésus-Christ, Saint-Esprit, CMCI, CMFI",  # termes/noms à orthographier correctement dans les textes produits
    CHAPTER_TITLE_MAX=50,  # longueur max des titres de chapitres (YouTube, noms des fichiers vidéo)
    MERGE_CONTINUING=True,  # fusionner les sections consécutives qui traitent du même thème (<= MAX_SECTION_SEC)
    THEME_SIM_CONTINUE=0.88,  # similarité (0-1) entre deux thèmes consécutifs au-delà de laquelle c'est une suite, même si le LLM dit non
    LLM_MAX_CHARS=20000,  # texte maximal envoyé au LLM par section (début + fin si plus long)

    LLM_HF_MODEL="Qwen/Qwen2.5-7B-Instruct",  # backend "transformers" (4 bits sur GPU)
    LLM_4BIT=True,
    LLM_BASE_URL="http://localhost:11434/v1",  # backend "openai" : Ollama ; LM Studio : http://localhost:1234/v1
    LLM_MODEL="qwen2.5-7b-12k",  # = qwen2.5:7b-instruct avec contexte 12k (voir README : `ollama create qwen2.5-7b-12k -f Modelfile`). Le contexte par défaut d'Ollama (4k) tronquerait les sections.
    LLM_API_KEY="",  # vide pour un serveur local (sinon variable/secret OPENAI_API_KEY possible)
    ANTHROPIC_MODEL="claude-sonnet-5-5",  # backend "anthropic" : clé dans ANTHROPIC_API_KEY (variable ou secret Colab)
    ANTHROPIC_URL="https://api.anthropic.com/v1/messages",

    # --- Exports ---------------------------------------------------------------
    EXPORT_FULL_TEXT=True,  # transcription_structuree.md (texte complet par section)
    CUT_SECTIONS=True,  # un fichier vidéo par section
    CUT_MODE="reencode",  # "reencode" (précis, plus lent) | "copy" (rapide, précis à quelques secondes près)
    EMBED_CHAPTERS=True,  # copie de la vidéo avec chapitres intégrés
)

## 2. Environnement

Installe les paquets manquants, vérifie ffmpeg, détecte le GPU et crée les dossiers de travail (`outputs/`) et de téléchargement (`data/`).
Sur Colab, cette cellule récupère d'abord le code du dépôt (`git clone`).

In [ ]:
import os, subprocess, sys

# Colab : récupérer le code du dépôt (en local, le dossier du notebook contient déjà `pipeline/`)
if "google.colab" in sys.modules and not os.path.isdir("pipeline"):
    REPO_URL = "https://github.com/1glls/socialMediaContents.git"
    if not os.path.isdir("/content/socialMediaContents"):
        subprocess.check_call(["git", "clone", "-q", REPO_URL, "/content/socialMediaContents"])
    os.chdir("/content/socialMediaContents")
sys.path.insert(0, os.getcwd())

from pipeline.environment import setup_environment

env = setup_environment(cfg)

## 3. Source et audio

- **Fichier** : utilisé tel quel. **Dossier** : toutes les vidéos qu'il contient (ex. une playlist déjà téléchargée dans `data/`).
- **URL YouTube** : téléchargée avec `yt-dlp` dans `DOWNLOAD_DIR` (`data/`), fichiers nommés d'après leur titre (1080p maximum, sinon le format inférieur le plus proche).
  Une **playlist** va dans `data/<titre de la playlist>/<NN> - <titre>.mp4` ; `PLAYLIST_ITEMS = "1-3"` limite aux premiers éléments.
  Une URL `watch?v=..&list=..` ne télécharge que la vidéo, sauf si `PLAYLIST = True`. YouTube bloque souvent les IP de Colab : télécharge alors en local.
- `INPUT = ""` sur Colab ouvre une fenêtre de téléversement.
- Si `INPUT` pointe vers un `audio.wav` produit par une exécution précédente, son dossier devient le dossier du projet.
- Chaque vidéo a son dossier de résultats : `outputs/<titre>/`, ou `outputs/<playlist>/<NN-titre>/` pour une playlist. Une vidéo en échec n'arrête pas les suivantes.
- `TRANSCRIPT_FILE` (sous-titre `.vtt`) ne s'applique qu'à une seule vidéo.

In [ ]:
from pipeline.source import resolve_sources

projects = resolve_sources(cfg, env)

## 4. Transcription

Priorité : `transcript.json` existant (cache) → import `TRANSCRIPT_FILE` (`.vtt` Teams/Zoom ou `.srt`) → faster-whisper.

Sur GPU, si les bibliothèques CUDA sont introuvables (erreur `libcublas` / `libcudnn` fréquente sur Colab), le notebook les précharge,
installe au besoin les paquets `nvidia-*-cu12`, puis retombe sur le CPU en dernier recours.

In [ ]:
from pipeline.transcription import transcribe

transcribe(cfg, env, projects)

## 5. Frontières des sections

1. Le texte est regroupé en **blocs de `BLOCK_SEC` secondes**, puis vectorisé.
2. Pour chaque point de coupure possible, on calcule la ressemblance entre `TOPIC_WINDOW_SEC` avant et après ; un **creux** de ressemblance indique un changement de sujet.
3. On retient les creux les plus nets, **au plus un toutes les `MIN_SECTION_SEC`**, jusqu'à obtenir une section tous les `TARGET_SECTION_SEC` environ. Une section dépassant `MAX_SECTION_SEC` est coupée de force à son meilleur creux.
4. Chaque frontière est ensuite **calée sur la pause ou la fin de phrase la plus proche** (± `BLOCK_SEC`).

Le LLM pourra ensuite **fusionner** deux sections consécutives qui traitent du même thème (étape 6).

## 6. Thèmes et descriptions rédigés par un LLM

Choix du moteur (`LLM_BACKEND`) :

- `auto` : GPU détecté (Colab) → modèle open source `transformers` ; sinon Ollama s'il répond sur `LLM_BASE_URL` ; sinon pas de LLM.
- `transformers` : Qwen2.5-7B-Instruct en 4 bits sur le GPU (premier téléchargement d'environ 15 Go).
- `openai` : tout serveur compatible OpenAI (Ollama, LM Studio, llama.cpp, services cloud). `anthropic` : API Anthropic.
  **Ces deux options cloud envoient le texte de la réunion à un tiers** : à choisir en connaissance de cause.

Pour chaque section, le LLM reçoit le texte et les **thèmes précédents**, et renvoie un thème (une phrase), un **titre court** de chapitre
(≤ `CHAPTER_TITLE_MAX` caractères), une description et indique si elle **prolonge directement la section précédente**
(fusion, ou « partie 1/2, 2/2 » si la fusion dépasse `MAX_SECTION_SEC`). Chaque section garde son propre thème.
Sans LLM, un résumé extractif prend le relais.

In [ ]:
from pipeline.themes import build_sections

build_sections(cfg, env, projects)

## 7. Texte complet par section (optionnel)

Écrit `transcription_structuree.md` : le plan, puis le texte de chaque section avec des repères de temps.

In [ ]:
from pipeline.exports import export_full_text

export_full_text(cfg, projects)

## 8. Découpage de la vidéo et chapitres

À partir de `sections.json`, les sections deviennent **contiguës** (la première commence à 0, chaque section s'arrête là où la suivante commence), donc la vidéo est entièrement couverte, sans trou ni chevauchement.

Sont produits :
- `sections_video/NN_<thème>.mp4` : un fichier par section (`CUT_MODE = "reencode"` : coupe exacte ; `"copy"` : instantané mais calé sur les images clés) ;
- `chapitres_youtube.txt` (titres courts) : à coller dans la description d'une vidéo YouTube (chapitres automatiques) ;
- `chapitres.ffmetadata` et `<nom>_chapitres.mp4` : la vidéo d'origine avec chapitres intégrés (lecteurs VLC, etc.), sans ré-encodage.

Pour un **flux en direct**, enregistre-le d'abord (OBS, `yt-dlp --live-from-start`...) puis fournis le fichier obtenu.

In [ ]:
from pipeline.video import cut_video

cut_video(cfg, env, projects)

In [ ]:
from pipeline.exports import show_results

show_results(cfg, env, projects)

## Réglages et dépannage

| Constat | Réglage |
|---|---|
| Sections trop courtes / trop nombreuses | Augmente `TARGET_SECTION_SEC` (ex. 600), `MIN_SECTION_SEC` ou `TOPIC_STRICTNESS` (ex. 0.5), puis `FORCE = {"themes"}`. |
| Une notion longue est coupée | Garde `MERGE_CONTINUING = True` (le LLM fusionne les sections qui se suivent sur le même thème, jusqu'à `MAX_SECTION_SEC`) ; augmente `MAX_SECTION_SEC`. |
| Thèmes encore peu clairs | Utilise un LLM plus fort : `LLM_HF_MODEL = "Qwen/Qwen2.5-14B-Instruct"` (GPU), un modèle plus gros dans Ollama, ou le backend `anthropic`. Renseigne `GLOSSARY`. |
| Noms propres mal transcrits | Renseigne `INITIAL_PROMPT` puis `FORCE = {"transcribe"}`. |
| Colab : erreur `libcublas` / `libcudnn` | Le notebook tente de les charger et de les installer ; sinon il bascule sur CPU. Redémarre l'environnement et relance si besoin. |
| Colab : mémoire GPU insuffisante | Whisper puis le LLM sont chargés l'un après l'autre ; réduis le modèle (`LLM_HF_MODEL = "Qwen/Qwen2.5-3B-Instruct"`). |
| Aucun LLM disponible en local | Installe Ollama et fais `ollama pull qwen2.5:7b-instruct`, ou utilise le résumé extractif (thème = phrase la plus représentative). |
| Coupes vidéo décalées de quelques secondes | Utilise `CUT_MODE = "reencode"` (coupe exacte). |

## Limites

- Les frontières sont des **changements de sujet détectés automatiquement** (précision de l'ordre de la pause la plus proche) : relis le plan, puis ajuste les paramètres.
- Un LLM de 7 milliards de paramètres peut reformuler imparfaitement ou rater une fusion ; la relecture reste nécessaire.
- La qualité dépend de la transcription (bruit, accents, mélange de langues).